In [0]:

-- This query does not modify the source file.
SELECT *
FROM read_files(
  '/Volumes/workspace/sentinel_sources/originals/fraudTrain.csv',
  -- Use the CSV header as column names.
  format => 'csv',
  header => true,
  -- Keep values as strings so identifiers are not converted to numbers.
  inferColumnTypes => false
)
-- Inspect five records from the original training file.
LIMIT 5;

In [0]:
-- Inspect the test file's columns and rescued data.
-- Compare its structure with the training file.
-- Preserve source values as strings during assessment.

SELECT *
FROM read_files(
  '/Volumes/workspace/sentinel_sources/originals/fraudTest.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false
)
LIMIT 5;

The preview gives us our first finding: _rescued_data contains an extra field named _c0, with values 0–4 in these five records.

That looks like an exported row index, but we’ll verify it before deciding how to handle it. A populated rescued column does not automatically mean the transaction is invalid

In [0]:
-- Count records and rows containing rescued fields in each file.
-- Rescued fields require investigation; they are not rejected here.

SELECT
  _metadata.file_name AS source_file,
  COUNT(*) AS row_count,
  COUNT_IF(_rescued_data IS NOT NULL) AS rows_with_rescued_data
FROM read_files(
  '/Volumes/workspace/sentinel_sources/originals/*.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false
)
GROUP BY _metadata.file_name
ORDER BY source_file;

Both previews have matching columns. total 1,852,394 records:
File	Records	With rescued data
fraudTrain.csv	1,296,675	1,296,675
fraudTest.csv	555,719	555,719


Every record has rescued data. The previews suggest an unnamed index column, so we’ll inspect the rescued keys before treating any rows as invalid.

Add these two SQL cells to 01_source_assessment.

In [0]:
-- List the combinations of keys present in rescued data.
-- This checks whether all rows contain only _c0 and _file_path,
-- or whether other unexpected fields also occur.

SELECT
  _metadata.file_name AS source_file,
  array_sort(
    map_keys(
      from_json(_rescued_data, 'MAP<STRING,STRING>')
    )
  ) AS rescued_keys,
  COUNT(*) AS row_count
FROM read_files(
  '/Volumes/workspace/sentinel_sources/originals/*.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false
)
GROUP BY source_file, rescued_keys
ORDER BY source_file, row_count DESC;

In [0]:
-- Parse event dates for assessment without changing source files.
-- TIMESTAMP_NTZ preserves the written clock time without
-- assuming a timezone that the source has not established.
-- TRY_CAST returns NULL when a value cannot be parsed.

WITH source_dates AS (
  SELECT
    _metadata.file_name AS source_file,
    trans_date_trans_time,
    TRY_CAST(
      trans_date_trans_time AS TIMESTAMP_NTZ
    ) AS event_time
  FROM read_files(
    '/Volumes/workspace/sentinel_sources/originals/*.csv',
    format => 'csv',
    header => true,
    inferColumnTypes => false
  )
)
SELECT
  source_file,
  MIN(event_time) AS earliest_event,
  MAX(event_time) AS latest_event,
  COUNT_IF(
    trans_date_trans_time IS NULL
    OR TRIM(trans_date_trans_time) = ''
  ) AS missing_event_dates,
  COUNT_IF(
    trans_date_trans_time IS NOT NULL
    AND TRIM(trans_date_trans_time) <> ''
    AND event_time IS NULL
  ) AS unparseable_event_dates
FROM source_dates
GROUP BY source_file
ORDER BY source_file;

Confirmed two useful findings:
- Rescued data contains only _c0 and _file_path across both files. We found no other rescued fields; _c0 remains a likely exported index.
- All event dates parse successfully. The training file ends on June 21, 2020 at 12:13:37, and the test file starts 48 seconds later. Their date ranges don’t overlap.
Next, check whether trans_num can uniquely identify a transaction across both files.

In [0]:
-- Check transaction references across BOTH source files.
-- A usable transaction key must be present and unique.
-- TRIM is used for assessment only; source files remain unchanged.

WITH source_ids AS (
  SELECT
    NULLIF(TRIM(trans_num), '') AS transaction_id
  FROM read_files(
    '/Volumes/workspace/sentinel_sources/originals/*.csv',
    format => 'csv',
    header => true,
    inferColumnTypes => false
  )
)
SELECT
  COUNT(*) AS total_rows,
  COUNT_IF(transaction_id IS NULL) AS missing_transaction_ids,
  COUNT(DISTINCT transaction_id) AS distinct_transaction_ids,

  -- Count extra occurrences of non-missing IDs.
  COUNT(transaction_id)
    - COUNT(DISTINCT transaction_id) AS repeated_id_extra_rows
FROM source_ids;

In [0]:
-- Check required monitoring fields before defining Silver rules.
-- Keep card identifiers as strings.
-- TRY_CAST detects amounts that cannot be represented as decimals.
-- Counts can overlap: one row may fail more than one check.

SELECT
  _metadata.file_name AS source_file,
  COUNT_IF(
    cc_num IS NULL OR TRIM(cc_num) = ''
  ) AS missing_card_ids,
  COUNT_IF(
    merchant IS NULL OR TRIM(merchant) = ''
  ) AS missing_merchants,
  COUNT_IF(
    amt IS NULL OR TRIM(amt) = ''
  ) AS missing_amounts,
  COUNT_IF(
    amt IS NOT NULL
    AND TRIM(amt) <> ''
    AND TRY_CAST(amt AS DECIMAL(18, 2)) IS NULL
  ) AS unparseable_amounts,
  COUNT_IF(
    TRY_CAST(amt AS DECIMAL(18, 2)) <= 0
  ) AS non_positive_amounts
FROM read_files(
  '/Volumes/workspace/sentinel_sources/originals/*.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false
)
GROUP BY _metadata.file_name
ORDER BY source_file;

These checks pass:
- trans_num is present and unique across all 1,852,394 records, so we can adopt it as the transaction key.
- Card IDs, merchants, and amounts are present.
- All amounts pass our decimal conversion check and are positive.
We have no evidence requiring rows to be removed for these reasons. Silver will still enforce these checks for future arrivals.

Next, we’ll examine the relationship between the two time fields. A

In [0]:
-- Compare the written transaction time with the numeric time field.
-- Calculate the written time's seconds since the standard Unix epoch,
-- without introducing a timezone assumption.
-- Group the differences to detect a consistent offset or mixed behavior.
-- This is an assessment only: do not correct either field yet.

WITH parsed_times AS (
  SELECT
    _metadata.file_name AS source_file,
    TRY_CAST(
      trans_date_trans_time AS TIMESTAMP_NTZ
    ) AS event_time,
    TRY_CAST(unix_time AS BIGINT) AS source_unix_seconds
  FROM read_files(
    '/Volumes/workspace/sentinel_sources/originals/*.csv',
    format => 'csv',
    header => true,
    inferColumnTypes => false
  )
),
time_comparison AS (
  SELECT
    source_file,
    TIMESTAMPDIFF(
      SECOND,
      TIMESTAMP_NTZ '1970-01-01 00:00:00',
      event_time
    ) - source_unix_seconds AS offset_seconds
  FROM parsed_times
)
SELECT
  source_file,
  offset_seconds,
  COUNT(*) AS row_count
FROM time_comparison
GROUP BY source_file, offset_seconds
ORDER BY source_file, row_count DESC;

## Findings so far

- Total source records: 1,852,394.
- Training records: 1,296,675.
- Test records: 555,719.
- Both previews expose matching columns.
- Rescued fields contain only _c0 and _file_path.
- _c0 appears to be an exported index; its meaning is not yet confirmed.
- Event dates have no missing or unparseable values.
- Training and test event-date ranges do not overlap.
- trans_num is non-missing and unique across both files.
- Card IDs, merchants, and amounts have no missing values.
- Amounts pass DECIMAL(18,2) conversion and are positive.
- Time-field consistency and replay-period selection remain pending.

## Event-time decision

Use trans_date_trans_time as event time, parsed as TIMESTAMP_NTZ.
The source timezone is unspecified.

unix_time differs from the written dates by:
- 220,838,400 seconds for 924,311 training records.
- 220,924,800 seconds for 372,364 training records
  and all 555,719 test records.

The cause remains unverified. Preserve unix_time unchanged,
but exclude it from event ordering and monitoring windows.

Next, let’s assess a continuous six-week subset:

In [0]:
-- Count daily activity in a proposed six-week subset.
-- Include every transaction within the period to preserve histories.
-- Do not inspect fraud labels when choosing monitoring thresholds.
-- The upper boundary is exclusive, so February 11 is fully included.

WITH source_events AS (
  SELECT
    TRY_CAST(
      trans_date_trans_time AS TIMESTAMP_NTZ
    ) AS event_time,
    cc_num
  FROM read_files(
    '/Volumes/workspace/sentinel_sources/originals/fraudTrain.csv',
    format => 'csv',
    header => true,
    inferColumnTypes => false
  )
)
SELECT
  CAST(event_time AS DATE) AS event_date,
  COUNT(*) AS transaction_count,
  COUNT(DISTINCT cc_num) AS active_cards
FROM source_events
WHERE event_time >= TIMESTAMP_NTZ '2019-01-01 00:00:00'
  AND event_time < TIMESTAMP_NTZ '2019-02-12 00:00:00'
GROUP BY CAST(event_time AS DATE)
ORDER BY event_date;

## Selected replay scope

Source: fraudTrain.csv.

Include transactions from 2019-01-01 00:00:00 inclusive
to 2019-02-12 00:00:00 exclusive.

- Initial history: January 1–14 — 24,144 transactions.
- Rule development: January 15–28 — 23,731 transactions.
- Evaluation: January 29–February 11 — 24,059 transactions.
- Total: 71,934 transactions across 42 consecutive dates.

Selection is chronological and does not use fraud labels.
Keep fraudTest.csv preserved for possible later validation;
it is outside this prototype's selected replay scope.

Prepare daily replay batches while preserving original event dates.
Keep original files separate from replay inputs.

Evaluation batches must remain unreleased until rule thresholds
are frozen. Evaluation labels remain outside monitoring and AI context.
